## **Environment Setup**

In [0]:
display("Hello Databricks")

'Hello Databricks'

In [0]:
# Install Kaggle, %pip means Databrics uses magic commands
%pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# New package installed → restart runtime to activate properly.
# Databricks loads Python packages into memory.
# Restart refreshes environment.
%restart_python

In [0]:
#Verify Everything Works , After restart.Compute alive 
display("Databricks Ready")

'Databricks Ready'

In [0]:
# Configure Kaggle Credentials.Set Environment Variables.Now we connect Kaggle server.
import os

os.environ["KAGGLE_USERNAME"] = kaggle_creds["username"]
os.environ["KAGGLE_KEY"] = kaggle_creds["key"]

print("Kaggle Credentials Configured!")
# Kaggle CLI checks: Environment Variables ,before looking for .kaggle folder

Kaggle Credentials Configured!


In [0]:
#TEST CONNECTION,whether You connected Kaggle server successfully or not

!kaggle datasets list

ref                                                                  title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
amar5693/screen-time-sleep-and-stress-analysis-dataset               Screen Time, Sleep & Stress Analysis Dataset            787136  2026-02-13 06:56:18.757000           7359        144                1  
amar5693/student-performance-dataset                                 Student Performance Dataset                             177286  2026-02-12 06:04:44.613000           6382        101                1  
aliiihussain/amazon-sales-dataset                                    Amazon_Sales_Dataset                                   1297759  2026-02-01 11:37:12.353000           8855      

## **Unity Catalog Setup**

In [0]:
#STEP 1 — Create Schema (Database Namespace) You created: workspace.ecommerce Now your tables later will live inside ecommerce. Think like: Company folder.

spark.sql("""CREATE SCHEMA IF NOT EXISTS workspace.ecommerce""")  

DataFrame[]

In [0]:
#STEP 2 — Create Volume (Persistent Storage)Why Volume?--> Serverless notebook storage disappears.So we need to save data somewhere Persistent. Volume = permanent cloud disk.If Cluster dies.Data survives,So anything downloaded here is permanent.

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
# STEP 3 — Check Volume Exists 
display(dbutils.fs.ls("/Volumes/workspace/ecommerce/ecommerce_data"))

[]

## **Dataset Download **

In [0]:
# STEP 4 — Download Dataset From Kaggle , with %sh You switched from Spark/Python context to cluster shell (Linux environment).

%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [01:04<00:00, 71.7MB/s]


In [0]:
# STEP 5 - Extract Raw Dataset from ZIP file 

%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-7bcf5c5a-06db-4383-935a-6b nogroup 8.4G Feb 27 14:17 2019-Nov.csv
-rwxrwxrwx 1 spark-7bcf5c5a-06db-4383-935a-6b nogroup 5.3G Feb 27 14:20 2019-Oct.csv
-rwxrwxrwx 1 nobody                           nogroup 4.3G Feb 26 17:18 ecommerce-behavior-data-from-multi-category-store.zip


In [0]:
# STEP 6 — Remove ZIP file(optional clean)
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 nobody nogroup 8.4G Feb 27 14:17 2019-Nov.csv
-rwxrwxrwx 1 nobody nogroup 5.3G Feb 27 14:20 2019-Oct.csv


In [0]:
# STEP 7 — Restart Python Runtime ,Spark caches metadata.Restart ensures:new files recognized cleanly.

%restart_python

## **Dataset Loading**

In [0]:
# STEP 8 — Load Dataset, you load multi-million event ecommerce datasets
df = spark.read.csv(
"/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",
header=True,
inferSchema=True
)

df.show(5)

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [0]:
# STEP 9 — Check Dataset Size

print("Toral Events:",df.count())

# See we have 42.4 million events

Toral Events: 42448764


In [0]:
# STEP 10 — Check Schema,,You should see 9 columns.
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [0]:
# STEP 11 — Check Sample View

df.show(10, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:0

## **Data Eng For Bronz Layer**

In [0]:
#Load November Dataset
df_n = spark.read.csv(
"/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
header=True,
inferSchema=True
)

In [0]:

# Load October dataset
df = spark.read.csv(
"/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",
header=True,
inferSchema=True
)


In [0]:
# Combine Oct+Nov datasets As Later need for recommendation engine.ML training.Needs more behaviour diversity.

events = df.union(df_n)

In [0]:
# Check by counting 
print(events.count())

109950743


In [0]:
# Check Null columns
from pyspark.sql import functions as F

events.select([
F.count(F.when(F.col(c).isNull(),c)).alias(c)
for c in events.columns
]).show()

+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|   brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+
|         0|         0|         0|          0|     35413780|15331243|    0|      0|          12|
+----------+----------+----------+-----------+-------------+--------+-----+-------+------------+



## **Bronze Layer Foundation**

In [0]:
# Save Raw Bronze Table. this becomes single source of truth
"""But in new Databricks:Public DBFS is disabled for security.All writes must go to Unity Catalog storage.You already created a Volume.So instead of writing to /delta/..., you must write to: /Volumes/workspace/ecommerce/ecommerce_data/..."""

#Correct Way to Save Bronze as Managed unity Catalog Delta Table***
events.write.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.ecommerce.bronze_events_raw")

#This creates: Managed Delta Table ,Registered in catalog ,Queryable via SQL

In [0]:
# Verify, table successfully regisered or not
spark.sql("SHOW TABLES IN workspace.ecommerce").show()

+---------+-----------------+-----------+
| database|        tableName|isTemporary|
+---------+-----------------+-----------+
|ecommerce|bronze_events_raw|      false|
+---------+-----------------+-----------+



In [0]:
# Row count verification check
spark.sql("SELECT COUNT(*) FROM workspace.ecommerce.bronze_events_raw").show()

+---------+
| COUNT(*)|
+---------+
|109950743|
+---------+



In [0]:
# """Advanced Bronze Engineering Upgrade (Modern Production Thinking) Improve the bronze layer"""

# Convert event_time from string to timestamp
#because Time analysis  ML streaming later depends on thisbb
from pyspark.sql.functions import to_timestamp, current_timestamp, to_date

events_clean = events.withColumn(
"event_time",
to_timestamp("event_time")
)

# Add Ingestion Metadata ; Now dataset remembers: when pipeline ingested data Industry audit requirement.
# Banks and healthcare absolutely require this
events_clean = events_clean.withColumn(
"ingestion_time",
current_timestamp()
)

# Create Event Date Column Add a derived column from the timestamp This creates a daily partition key.
events_enhanced = events_clean.withColumn(
    "event_date",
    to_date("event_time")
)


# Remove Completely Empty Rows(Rare but happened 
# Not aggressive cleaning for just safety
events_clean = events_clean.dropna(
how="all"
) 

# Save CLEAN BRONZE as bronze_events Table With Partitioning ..This is textbook medallion architecture
events_enhanced.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .partitionBy("event_date") \
    .saveAsTable("workspace.ecommerce.bronze_events")


## Data Quality Check

In [0]:
#VERIFY
spark.sql("""
SHOW TABLES IN workspace.ecommerce
""").show()




+---------+-----------------+-----------+
| database|        tableName|isTemporary|
+---------+-----------------+-----------+
|ecommerce|    bronze_events|      false|
|ecommerce|bronze_events_raw|      false|
+---------+-----------------+-----------+



In [0]:
# Null distribution check
spark.sql("""
SELECT
COUNT(*) total_rows,
COUNT(brand) non_null_brand,
COUNT(category_code) non_null_category
FROM workspace.ecommerce.bronze_events
""").show()

+----------+--------------+-----------------+
|total_rows|non_null_brand|non_null_category|
+----------+--------------+-----------------+
| 109950743|      94619500|         74536963|
+----------+--------------+-----------------+



In [0]:
# schema validation check
spark.table(
"workspace.ecommerce.bronze_events"
).printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



In [0]:
# Time Range check 
spark.sql("""
SELECT
MIN(event_time) min_event_time,
MAX(event_time) max_event_time
FROM workspace.ecommerce.bronze_events
""").show()

+-------------------+-------------------+
|     min_event_time|     max_event_time|
+-------------------+-------------------+
|2019-10-01 00:00:00|2019-11-30 23:59:59|
+-------------------+-------------------+



In [0]:

# Event Distribution,This distribution reflects a typical e-commerce conversion funnel 
# Later stages will use this data to:build purchase prediction models,create user behavior features, train recommendation systems


spark.sql("""
SELECT event_type, COUNT(*) as events
FROM workspace.ecommerce.bronze_events
GROUP BY event_type
ORDER BY events DESC
""").show()

+----------+---------+
|event_type|   events|
+----------+---------+
|      view|104335509|
|      cart|  3955446|
|  purchase|  1659788|
+----------+---------+



## **Performance Engineering **

What Day-1 Actually Teaches Day-1 is about Delta performance optimization.

Concepts introduced: Delta vs Parquet,
small file problem,OPTIMIZE command,ZORDER indexing,query execution plans

The goal is to understand how Delta organizes files and how optimization improves query performance.

What We Will Do (Better Than Tutorial)
Instead of blindly following the tutorial, we will do a proper system experiment.
We will:
1️⃣ Inspect current file layout
2️⃣ Create small file fragmentation
3️⃣ Measure query behavior
4️⃣ Apply Delta optimization
5️⃣ Observe improvement
This teaches the real purpose of OPTIMIZE.

In [0]:
# Inspect Current Table Layout:--

spark.sql("""
DESCRIBE DETAIL workspace.ecommerce.bronze_events
""").show(truncate=False)

+------+------------------------------------+---------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                             |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                    |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------------+---------------------------------+-----------+--------+-----------------------+--------

In [0]:
# Inspect Delta History:--

spark.sql("""
DESCRIBE HISTORY workspace.ecommerce.bronze_events
""").show(truncate=False)

+-------+-------------------+--------------+---------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                   |operation                        |operationParameters                                                                                                                                                                                         |job |notebook          |queryHis

In [0]:
# Explain Query Plan:--

spark.sql("""
SELECT *
FROM workspace.ecommerce.bronze_events
WHERE event_type = 'purchase'
""").explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('event_type = purchase)
   +- 'UnresolvedRelation [workspace, ecommerce, bronze_events], [], false

== Analyzed Logical Plan ==
event_time: timestamp, event_type: string, product_id: int, category_id: bigint, category_code: string, brand: string, price: double, user_id: int, user_session: string, ingestion_time: timestamp, event_date: date
Project [event_time#18374, event_type#18375, product_id#18376, category_id#18377L, category_code#18378, brand#18379, price#18380, user_id#18381, user_session#18382, ingestion_time#18383, event_date#18384]
+- Filter (event_type#18375 = purchase)
   +- SubqueryAlias workspace.ecommerce.bronze_events
      +- Relation workspace.ecommerce.bronze_events[event_time#18374,event_type#18375,product_id#18376,category_id#18377L,category_code#18378,brand#18379,price#18380,user_id#18381,user_session#18382,ingestion_time#18383,event_date#18384] parquet

== Optimized Logical Plan ==
Filter (isnotnull(event_type#183

Small File Problem

In real pipelines, streaming or batch ingestion can produce thousands of tiny files, which slows queries dramatically.

Delta provides tools to fix this:

OPTIMIZE
ZORDER

We will simulate that scenario.

## **Create Small Files (Learning Experiment)**

In [0]:
# Step 1 —  Run this to intentionally pollutes the dataset by creating fragmentation .This simulates a streaming pipeline producing small batches.. Think of it like:software unit testing to observe behaviour
for i in range(10):
    spark.table("workspace.ecommerce.bronze_events") \
        .limit(1000) \
        .write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.ecommerce.bronze_events")

In [0]:
# Step 2 — Check File Count Again, You should see numFiles increase significantly.

spark.sql("""
DESCRIBE DETAIL workspace.ecommerce.bronze_events
""").show(truncate=False)

+------+------------------------------------+---------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                             |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                    |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------------+---------------------------------+-----------+--------+-----------------------+--------

In [0]:
# Step 3 — Fix the Problem Using OPTIMIZE, This merges many small files into larger optimized files.
spark.sql("""
OPTIMIZE workspace.ecommerce.bronze_events
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# Step 4 — Improve Query Skipping, Using ZORDER by event_type,ZORDER clusters similar values together. Now Spark finds faster.  Huge optimization. WHERE event_type = 'purchase' kind of  Targated scan possible:
spark.sql("""
OPTIMIZE workspace.ecommerce.bronze_events
ZORDER BY (event_type)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# Let’s verify instead of resetting blindly.If your row count is still around 110M, then your dataset is fine. No Reset required for next stage of clean pipeline buildup

spark.sql("""
SELECT COUNT(*)
FROM workspace.ecommerce.bronze_events
""").show()


+---------+
| COUNT(*)|
+---------+
|109950743|
+---------+



In [0]:
# Before building Silver, do one final Bronze validation.Confirm the counts still look realistic
spark.sql("""
SELECT event_type, COUNT(*)
FROM workspace.ecommerce.bronze_events
GROUP BY event_type
""").show()

+----------+---------+
|event_type| COUNT(*)|
+----------+---------+
|  purchase|  1659788|
|      cart|  3955446|
|      view|104335509|
+----------+---------+



In [0]:
#One last Bronze inspection (optional but useful)
spark.sql("""
SELECT
MIN(event_time) as first_event,
MAX(event_time) as last_event
FROM workspace.ecommerce.bronze_events
""").show()

+-------------------+-------------------+
|        first_event|         last_event|
+-------------------+-------------------+
|2019-10-01 00:00:00|2019-11-30 23:59:59|
+-------------------+-------------------+



## **Delta Lake Features**

In [0]:
spark.sql("""
DESCRIBE HISTORY workspace.ecommerce.bronze_events
""").show(truncate=False)

# if runs ok means _delta_log is present, table is truely Delta format
# Versioning is active, Transactions are being tracked.. 
# so things went properly

+-------+-------------------+--------------+---------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                   |operation                        |operationParameters                                                                                                          

In [0]:
old_df = spark.read.format("delta") \
.option("versionAsOf",0) \
.table("workspace.ecommerce.bronze_events")

old_df.show(5)

+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+--------------------+----------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|      ingestion_time|event_date|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+--------------------+----------+
|2019-10-22 00:00:00|      view|   5100337|2053013553341792533|  electronics.clocks| apple|321.33|562785027|96058743-5434-4a4...|2026-03-04 15:15:...|2019-10-22|
|2019-10-22 00:00:00|      view|  10400206|2053013553257906447|           kids.toys|  NULL|140.65|543949853|ede3685b-bb0e-48f...|2026-03-04 15:15:...|2019-10-22|
|2019-10-22 00:00:00|      view|  15100009|2053013557024391671|                NULL|  NULL|321.73|519206046|737aa876-bb46-4fe...|2026-03-04 15:15:...|2019-10-22|
|2019-10-22 00:00:00|      v